# Custom Indian Traffic Sign Recognition (TSR) Training Pipeline
### Trained on 2,338 Authentic Indian Road IRC Signboards (15 Canonical Classes)

This notebook trains **YOLOv8s** on your custom, clean Indian TSR dataset:
- **Zero clipart / flat white stickers** (filtered out)
- **Zero perceptual duplicates** (pHash filtered)
- **Zero class contamination** (IRC standard)
- **Strict directional safety** (`fliplr=0.0` so Left Turn and Right Turn remain distinct)

---

### Step 1: Verify GPU & Install Ultralytics

In [ ]:
!nvidia-smi
!pip install --upgrade ultralytics matplotlib pyyaml pillow

### Step 2: Upload and Extract `custom_indian_tsr.zip`
Upload `custom_indian_tsr.zip` (507 MB) directly from your `TrafficSign/dataset` folder.

In [ ]:
import os, zipfile
from google.colab import files

if not os.path.exists("custom_indian_tsr.zip"):
    print("Please choose 'custom_indian_tsr.zip' from your TrafficSign/dataset folder:")
    uploaded = files.upload()

print("Extracting dataset archive...")
with zipfile.ZipFile("custom_indian_tsr.zip", "r") as zip_ref:
    zip_ref.extractall("custom_indian_tsr")

print("Dataset extracted successfully to custom_indian_tsr/")

### Step 3: Verify Dataset YAML & Classes

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path("custom_indian_tsr/dataset.yaml")
with open(yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = str(Path("/content/custom_indian_tsr").resolve())
with open(yaml_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print("=== DATASET CONFIGURATION ===")
print(f"Base Path: {cfg['path']}")
print(f"Number of classes: {cfg['nc']}")
for cid, name in cfg["names"].items():
    print(f"  Class {cid:2d}: {name}")

### Step 4: Train YOLOv8s (Clean Indian TSR)
*Notice: `fliplr=0.0` preserves Left Turn vs Right Turn integrity.*

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data="custom_indian_tsr/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    fliplr=0.0,      # Disabled so Left and Right turns do not mirror
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    project="runs/detect",
    name="indian_tsr_custom",
    save=True,
    plots=True
)

### Step 5: Validate and Inspect Confusion Matrix

In [ ]:
from PIL import Image
from IPython.display import display
from pathlib import Path

cm_path = Path("runs/detect/indian_tsr_custom/confusion_matrix.png")
if cm_path.exists():
    display(Image.open(cm_path))

results_path = Path("runs/detect/indian_tsr_custom/results.png")
if results_path.exists():
    display(Image.open(results_path))

### Step 6: Download the New `best.pt` Weights

In [ ]:
from google.colab import files
from pathlib import Path

best_weights = Path("runs/detect/indian_tsr_custom/weights/best.pt")
if best_weights.exists():
    mb = best_weights.stat().st_size / (1024 * 1024)
    print(f"Downloading best.pt ({mb:.2f} MB)...")
    files.download(str(best_weights))
else:
    print("Error: best.pt weights file not found!")